In [24]:
import os
import glob
import pandas as pd
import pdfplumber

In [25]:
raw_dir = "../raw"
clean_dir = "../clean"

def extract_pdf(pdf_path):
    dfs = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            table = page.extract_table()
            if table:
                df = pd.DataFrame(table[1:], columns=table[0])
                dfs.append(df)
    return dfs

def process_raw():
    search = os.path.join(raw_dir,"*")
    all_files = glob.glob(search)
    count_total = 0
    count_fail = 0
    
    if not all_files:
        print(f"No files found in '{raw_dir}' folder.")
        return
    for file_path in all_files:
       # file_path = os.path.basename(file_path) -------DELETE
        count_total += 1
        if os.path.isdir(file_path):
            continue

        file_name = os.path.basename(file_path)
        ext = file_name.split('.')[-1].lower()
        base_name = os.path.splitext(file_name)[0]
        completed = round(100*(1-count_fail/count_total),2)
        print(f"Processing: {file_name}... %",completed, flush=True)

        #Check for if file is already "clean"
        clean_path = os.path.join(clean_dir, f"clean_{base_name}.csv")
        if os.path.exists(clean_path): continue
        
        try:
            if ext == 'csv':
                df = pd.read_csv(file_path)
                df.to_csv(clean_path, index=False)
            
            elif ext in ['xlsx','xlsm']:
                df = pd.read_excel(file_path, engine='openpyxl')
                df.to_csv(clean_path, index=False)

            elif ext == 'xls':
                df = pd.read_excel(file_path, engine='xlrd')
                df.to_csv(clean_path, index=False)

            elif ext == 'pdf':
                pdf_df = extract_pdf(file_path)
                if pdf_df:
                    combined_pdf_df = pd.concat(pdf_df, ignore_index=True)
                    combined_pdf_df.to_csv(clean_path, index=False)
                else:
                    print(f"!!!!! No tabular data could be extracted from PDF: {file_name}")
                    count_fail += 1
            else:
                print(f"No supported file: {file_name}")
                count_fail += 1
        except Exception as x:
            print(f"ERROR! processing {file_name}>>>> {x}")
            count_fail += 1   

process_raw()
print("\nCleaning Completed")                    
        

Processing: a00147_infeco_definitivo2026-03.pdf... % 100.0
ERROR! processing a00147_infeco_definitivo2026-03.pdf>>>> Reindexing only valid with uniquely valued Index objects
Processing: a00149_TMC4001.pdf... % 50.0
Processing: a00150_tasaus_mc.pdf... % 66.67
!!!!! No tabular data could be extracted from PDF: a00150_tasaus_mc.pdf
Processing: a00153_ipc_base_2019-2020.xls... % 50.0
Processing: a00168_economia_int2026-06.pdf... % 60.0
ERROR! processing a00168_economia_int2026-06.pdf>>>> Reindexing only valid with uniquely valued Index objects
Processing: a00170_Boletin_Trimestral_Mercado_Laboral_enero-marzo_2026.pdf... % 50.0
ERROR! processing a00170_Boletin_Trimestral_Mercado_Laboral_enero-marzo_2026.pdf>>>> Reindexing only valid with uniquely valued Index objects
Processing: a00170_Informe-IPC-2026-06.pdf... % 42.86
ERROR! processing a00170_Informe-IPC-2026-06.pdf>>>> Reindexing only valid with uniquely valued Index objects
Processing: a00172_Esquema_de_Metas_de_Inflaci%C3%B3n.pdf... % 